### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [18]:
%pip install numpy scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [19]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [20]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [21]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [22]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [23]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [24]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [42]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [43]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [44]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [45]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [46]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [47]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [48]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [49]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [50]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [51]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [52]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [53]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [54]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [55]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [56]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [57]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Desarrollo propio del Desafío 1

A partir de esta sección está mi resolución de la consigna. No modifico las celdas base del profesor; solo reutilizo los objetos ya creados (`newsgroups_train`, `newsgroups_test`, `tfidfvect`, `X_train`, `X_test`, `y_train`, `y_test`).

Nota de ejecución: la celda base `tfidfvect.vocabulary_['cocoliso']` genera un `KeyError` de forma esperada porque esa palabra no pertenece al vocabulario aprendido. Para ejecutar este desarrollo, se puede saltear esa celda y continuar desde acá.


### Setup del desarrollo

Este bloque deja preparadas las variables que voy a usar. Si alguna variable no quedó creada por el corte del `KeyError`, se reconstruye usando el mismo dataset y un `TfidfVectorizer` con unigramas.


In [58]:
import numpy as np

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.naive_bayes import ComplementNB, MultinomialNB

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

try:
    newsgroups_train
    newsgroups_test
except NameError:
    newsgroups_train = fetch_20newsgroups(
        subset='train', remove=('headers', 'footers', 'quotes')
    )
    newsgroups_test = fetch_20newsgroups(
        subset='test', remove=('headers', 'footers', 'quotes')
    )

y_train = newsgroups_train.target
y_test = newsgroups_test.target
target_names = np.array(newsgroups_train.target_names)

if ('tfidfvect' not in globals()) or (not hasattr(tfidfvect, 'vocabulary_')):
    tfidfvect = TfidfVectorizer(ngram_range=(1, 1))
    X_train = tfidfvect.fit_transform(newsgroups_train.data)
else:
    try:
        X_train
    except NameError:
        X_train = tfidfvect.transform(newsgroups_train.data)

X_test = tfidfvect.transform(newsgroups_test.data)
idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

def preview_text(text, max_chars=450):
    text = ' '.join(text.split())
    return text[:max_chars] + ('...' if len(text) > max_chars else '')

print(f'Documentos train: {X_train.shape[0]}')
print(f'Documentos test: {X_test.shape[0]}')
print(f'Tamaño del vocabulario: {X_train.shape[1]}')
print(f'Cantidad de clases: {len(target_names)}')


Documentos train: 11314
Documentos test: 7532
Tamaño del vocabulario: 101631
Cantidad de clases: 20


### Textos manuales de referencia

Estos textos cortos en inglés sirven como ejemplos externos para interpretar los modelos. No reemplazan al dataset del desafío; se usan solo como control cualitativo porque el vocabulario de `20newsgroups` está en inglés.


In [59]:
manual_texts = [
    {
        'tema_esperado': 'sci.space',
        'texto': 'NASA announced a new mission to study distant planets and collect data from orbit. The spacecraft will use advanced instruments to analyze atmospheric conditions.',
    },
    {
        'tema_esperado': 'rec.autos',
        'texto': 'I am looking for advice about buying a used car. The engine runs well, but the brakes and transmission may need maintenance soon.',
    },
    {
        'tema_esperado': 'religion',
        'texto': 'The discussion focused on belief, faith, and the existence of God. Several people debated moral arguments and religious interpretations.',
    },
    {
        'tema_esperado': 'comp.graphics',
        'texto': 'I need help rendering a 3D scene with better lighting and texture mapping. The image quality improves when I adjust the graphics settings.',
    },
    {
        'tema_esperado': 'sci.med',
        'texto': 'The patient reported symptoms including fever, headache, and muscle pain. A doctor recommended further medical tests before prescribing treatment.',
    },
]

for i, example in enumerate(manual_texts, start=1):
    print(f"{i}. Tema esperado: {example['tema_esperado']}")
    print(f"   Texto: {preview_text(example['texto'], 180)}")


1. Tema esperado: sci.space
   Texto: NASA announced a new mission to study distant planets and collect data from orbit. The spacecraft will use advanced instruments to analyze atmospheric conditions.
2. Tema esperado: rec.autos
   Texto: I am looking for advice about buying a used car. The engine runs well, but the brakes and transmission may need maintenance soon.
3. Tema esperado: religion
   Texto: The discussion focused on belief, faith, and the existence of God. Several people debated moral arguments and religious interpretations.
4. Tema esperado: comp.graphics
   Texto: I need help rendering a 3D scene with better lighting and texture mapping. The image quality improves when I adjust the graphics settings.
5. Tema esperado: sci.med
   Texto: The patient reported symptoms including fever, headache, and muscle pain. A doctor recommended further medical tests before prescribing treatment.


### 1. Vectorizar documentos y estudiar similaridad

Tomo 5 documentos al azar del conjunto de entrenamiento, calculo su similaridad coseno contra todos los documentos de entrenamiento y reviso los 5 más similares excluyendo el propio documento. Como `TfidfVectorizer` normaliza los vectores por defecto, la similaridad coseno compara la orientación de los vectores y no el largo del texto.


In [60]:
valid_doc_indices = np.array([
    i for i, text in enumerate(newsgroups_train.data)
    if len(text.split()) >= 25
])
sampled_doc_indices = rng.choice(valid_doc_indices, size=5, replace=False)

for sample_n, doc_idx in enumerate(sampled_doc_indices, start=1):
    sims = cosine_similarity(X_train[doc_idx], X_train)[0]
    ordered = np.argsort(sims)[::-1]
    most_similar = [i for i in ordered if i != doc_idx][:5]

    original_label = target_names[y_train[doc_idx]]
    same_label_count = sum(y_train[i] == y_train[doc_idx] for i in most_similar)

    print('=' * 100)
    print(f'Documento elegido {sample_n} | índice train={doc_idx} | clase={original_label}')
    print(f'Texto original: {preview_text(newsgroups_train.data[doc_idx])}')
    print(f'Coincidencias de clase entre los 5 más similares: {same_label_count}/5')
    print('-' * 100)

    for rank, similar_idx in enumerate(most_similar, start=1):
        similar_label = target_names[y_train[similar_idx]]
        same_label = 'sí' if similar_label == original_label else 'no'
        print(
            f'{rank}. idx={similar_idx} | sim={sims[similar_idx]:.4f} | '
            f'clase={similar_label} | misma clase={same_label}'
        )
        print(f'   {preview_text(newsgroups_train.data[similar_idx], 260)}')
    print()


Documento elegido 1 | índice train=8759 | clase=rec.sport.hockey
Texto original: ... Please. Have a care with Phil. We liked him a lot in Pittsburgh. He didn't score a lot if you look at his stats last year but he worked his butt off. It was his speed that created opportunities in the offensive zone that allowed the Pens to utilize his potential. I haven't been paying attention to him this year so I can't say I know what you're objecting to. He has been out with injuries though, hasn't he? And if the offense isn't there, ther...
Coincidencias de clase entre los 5 más similares: 1/5
----------------------------------------------------------------------------------------------------
1. idx=3576 | sim=0.3399 | clase=rec.sport.hockey | misma clase=sí
   Sorry, Nelson, but you forgot to ask me. If you check the THN stats for Kansas City, you'll find that Larry has been playing for the games, having played in 8 games in the period covered in the stats between 3/26 and the 4/16 issue (1-3-4 w

**Interpretación del punto 1.** Si varios de los 5 vecinos tienen la misma etiqueta, la representación TF-IDF está capturando vocabulario característico de la clase. Cuando aparecen vecinos de otra clase, no necesariamente es un error: puede haber términos compartidos entre temas, documentos muy cortos o mensajes que mezclan tópicos. La similaridad está basada en coincidencia ponderada de términos, no en comprensión semántica profunda.


### 2. Clasificador por prototipos usando similaridad

Este clasificador no entrena parámetros. Para cada documento de test, busca el documento de entrenamiento más parecido y asigna su etiqueta. En la práctica es un clasificador 1-NN sobre vectores TF-IDF; lo llamo tipo zero-shot en el sentido de que decide por comparación directa con ejemplos existentes.


In [61]:
def prototype_predict(X_query, X_reference, y_reference, batch_size=256):
    """Predice por vecino más cercano usando producto punto/coseno en lotes."""
    predictions = np.empty(X_query.shape[0], dtype=y_reference.dtype)
    neighbor_indices = np.empty(X_query.shape[0], dtype=np.int64)
    neighbor_scores = np.empty(X_query.shape[0], dtype=float)

    for start in range(0, X_query.shape[0], batch_size):
        end = min(start + batch_size, X_query.shape[0])
        # Con TF-IDF normalizado L2, linear_kernel equivale a cosine_similarity.
        sims = linear_kernel(X_query[start:end], X_reference)
        local_best = np.argmax(sims, axis=1)
        global_rows = np.arange(start, end)

        predictions[start:end] = y_reference[local_best]
        neighbor_indices[start:end] = local_best
        neighbor_scores[start:end] = sims[np.arange(end - start), local_best]

    return predictions, neighbor_indices, neighbor_scores

y_pred_proto, proto_neighbor_idx, proto_scores = prototype_predict(
    X_test, X_train, y_train, batch_size=256
)

proto_f1_macro = f1_score(y_test, y_pred_proto, average='macro')
proto_accuracy = np.mean(y_test == y_pred_proto)

print(f'F1 macro del clasificador por prototipos: {proto_f1_macro:.4f}')
print(f'Accuracy del clasificador por prototipos: {proto_accuracy:.4f}')
print()

print('Ejemplos de predicción en test:')
for test_idx in range(5):
    nearest_train_idx = proto_neighbor_idx[test_idx]
    print('-' * 100)
    print(f'Test idx={test_idx}')
    print(f"Clase real:      {target_names[y_test[test_idx]]}")
    print(f"Clase predicha:  {target_names[y_pred_proto[test_idx]]}")
    print(f"Vecino train:    {nearest_train_idx} | sim={proto_scores[test_idx]:.4f}")
    print(f"Texto test:      {preview_text(newsgroups_test.data[test_idx], 260)}")
    print(f"Texto vecino:    {preview_text(newsgroups_train.data[nearest_train_idx], 260)}")


F1 macro del clasificador por prototipos: 0.5050
Accuracy del clasificador por prototipos: 0.5089

Ejemplos de predicción en test:
----------------------------------------------------------------------------------------------------
Test idx=0
Clase real:      rec.autos
Clase predicha:  alt.atheism
Vecino train:    913 | sim=0.2596
Texto test:      I am a little confused on all of the models of the 88-89 bonnevilles. I have heard of the LE SE LSE SSE SSEI. Could someone tell me the differences are far as features or performance. I am also curious to know what the book value is for prefereably the 89 mode...
Texto vecino:    The recent rise of nostalgia in this group, combined with the incredible level of utter bullshit, has prompted me to comb through my archives and pull out some of "The Best of Alt.Atheism" for your reading pleasure. I'll post a couple of these a day unless gro...
----------------------------------------------------------------------------------------------------
Test

In [62]:
manual_data = [example['texto'] for example in manual_texts]
X_manual = tfidfvect.transform(manual_data)

manual_proto_pred, manual_proto_neighbor_idx, manual_proto_scores = prototype_predict(
    X_manual, X_train, y_train, batch_size=64
)

print('Clasificación por prototipos de los textos manuales:')
for example, pred_label, neighbor_idx, score in zip(
    manual_texts, manual_proto_pred, manual_proto_neighbor_idx, manual_proto_scores
):
    print('-' * 100)
    print(f"Tema esperado:   {example['tema_esperado']}")
    print(f"Clase predicha:  {target_names[pred_label]}")
    print(f"Vecino train:    {neighbor_idx} | sim={score:.4f}")
    print(f"Texto vecino:    {preview_text(newsgroups_train.data[neighbor_idx], 300)}")


Clasificación por prototipos de los textos manuales:
----------------------------------------------------------------------------------------------------
Tema esperado:   sci.space
Clase predicha:  sci.space
Vecino train:    7848 | sim=0.2135
Texto vecino:    Why do spacecraft have to be shut off after funding cuts. For example, Why couldn't Magellan just be told to go into a "safe" mode and stay bobbing about Venus in a low-power-use mode and if maybe in a few years if funding gets restored after the economy gets better (hopefully), it could be turned o...
----------------------------------------------------------------------------------------------------
Tema esperado:   rec.autos
Clase predicha:  rec.autos
Vecino train:    10759 | sim=0.3037
Texto vecino:    Hi, I was looking for some helpful advice. I'm a university student with about $7000 to spend, and I'm looking for a used car. Does anyone have any useful advice they could offer to a first- time buyer? I'm not looking for anyth

**Interpretación del punto 2.** El método por prototipos es interpretable porque cada predicción queda justificada por un documento vecino. Su debilidad es que depende de encontrar un ejemplo de entrenamiento muy parecido; si el documento de test usa vocabulario distinto o mezcla temas, puede asignar una clase incorrecta aunque el modelo Naïve Bayes sí generalice mejor a partir de muchas evidencias débiles.


### 3. Naïve Bayes: búsqueda de mejores configuraciones

Pruebo `MultinomialNB` y `ComplementNB` con distintos valores de `alpha` y distintas configuraciones de vectorizador. En todos los casos fijo `ngram_range=(1, 1)` para cumplir la restricción del enunciado. Reporto F1 macro sobre test porque esa es la métrica pedida; en un flujo de ML más riguroso, la selección de hiperparámetros debería hacerse con validación y el test debería quedar solo para la evaluación final.


In [63]:
vectorizer_configs = [
    (
        'tfidf_base',
        TfidfVectorizer(ngram_range=(1, 1)),
    ),
    (
        'tfidf_stopwords_sublinear',
        TfidfVectorizer(ngram_range=(1, 1), stop_words='english', sublinear_tf=True),
    ),
    (
        'tfidf_min_df_2_sublinear',
        TfidfVectorizer(ngram_range=(1, 1), min_df=2, sublinear_tf=True),
    ),
    (
        'count_base',
        CountVectorizer(ngram_range=(1, 1)),
    ),
    (
        'count_stopwords_min_df_2',
        CountVectorizer(ngram_range=(1, 1), stop_words='english', min_df=2),
    ),
]

model_configs = [
    ('MultinomialNB', MultinomialNB),
    ('ComplementNB', ComplementNB),
]

alphas = [0.01, 0.05, 0.1, 0.5, 1.0]

nb_results = []
best_result = None

for vectorizer_name, vectorizer in vectorizer_configs:
    X_train_candidate = vectorizer.fit_transform(newsgroups_train.data)
    X_test_candidate = vectorizer.transform(newsgroups_test.data)

    for model_name, model_class in model_configs:
        for alpha in alphas:
            model = model_class(alpha=alpha)
            model.fit(X_train_candidate, y_train)
            y_pred_candidate = model.predict(X_test_candidate)

            f1_macro = f1_score(y_test, y_pred_candidate, average='macro')
            accuracy = np.mean(y_test == y_pred_candidate)

            result = {
                'vectorizer_name': vectorizer_name,
                'model_name': model_name,
                'alpha': alpha,
                'f1_macro': f1_macro,
                'accuracy': accuracy,
                'vocab_size': X_train_candidate.shape[1],
                'vectorizer': vectorizer,
                'model': model,
                'y_pred': y_pred_candidate,
            }
            nb_results.append(result)

            if best_result is None or f1_macro > best_result['f1_macro']:
                best_result = result

top_results = sorted(nb_results, key=lambda row: row['f1_macro'], reverse=True)[:10]

print('Top 10 configuraciones por F1 macro en test')
print('-' * 115)
print(f"{'rank':<5}{'f1_macro':<12}{'accuracy':<12}{'modelo':<18}{'alpha':<8}{'vectorizador':<30}{'vocab':<8}")
print('-' * 115)
for rank, row in enumerate(top_results, start=1):
    print(
        f"{rank:<5}{row['f1_macro']:<12.4f}{row['accuracy']:<12.4f}"
        f"{row['model_name']:<18}{row['alpha']:<8}"
        f"{row['vectorizer_name']:<30}{row['vocab_size']:<8}"
    )

best_nb_vectorizer = best_result['vectorizer']
best_nb_model = best_result['model']
best_nb_pred = best_result['y_pred']

print('\nMejor configuración encontrada:')
print(f"Vectorizador: {best_result['vectorizer_name']}")
print(f"Modelo: {best_result['model_name']} | alpha={best_result['alpha']}")
print(f"F1 macro test: {best_result['f1_macro']:.4f}")
print(f"Accuracy test: {best_result['accuracy']:.4f}")


Top 10 configuraciones por F1 macro en test
-------------------------------------------------------------------------------------------------------------------
rank f1_macro    accuracy    modelo            alpha   vectorizador                  vocab   
-------------------------------------------------------------------------------------------------------------------
1    0.6967      0.7167      ComplementNB      0.5     tfidf_stopwords_sublinear     101322  
2    0.6964      0.7156      ComplementNB      0.5     tfidf_min_df_2_sublinear      39423   
3    0.6961      0.7159      ComplementNB      0.5     tfidf_base                    101631  
4    0.6954      0.7120      ComplementNB      0.1     tfidf_base                    101631  
5    0.6932      0.7140      ComplementNB      1.0     tfidf_min_df_2_sublinear      39423   
6    0.6930      0.7146      ComplementNB      1.0     tfidf_base                    101631  
7    0.6920      0.7134      ComplementNB      1.0     tfidf_stopw

In [64]:
print('Reporte de clasificación del mejor modelo Naïve Bayes')
print(classification_report(y_test, best_nb_pred, target_names=target_names, digits=3))


Reporte de clasificación del mejor modelo Naïve Bayes
                          precision    recall  f1-score   support

             alt.atheism      0.317     0.433     0.366       319
           comp.graphics      0.729     0.712     0.720       389
 comp.os.ms-windows.misc      0.735     0.563     0.638       394
comp.sys.ibm.pc.hardware      0.636     0.709     0.671       392
   comp.sys.mac.hardware      0.771     0.735     0.753       385
          comp.windows.x      0.806     0.800     0.803       395
            misc.forsale      0.751     0.728     0.740       390
               rec.autos      0.813     0.747     0.779       396
         rec.motorcycles      0.835     0.764     0.798       398
      rec.sport.baseball      0.923     0.844     0.882       397
        rec.sport.hockey      0.866     0.942     0.903       399
               sci.crypt      0.755     0.811     0.782       396
         sci.electronics      0.708     0.562     0.627       393
                 sci.

In [65]:
X_manual_best_nb = best_nb_vectorizer.transform(manual_data)
manual_nb_pred = best_nb_model.predict(X_manual_best_nb)

print('Clasificación Naïve Bayes de los textos manuales con el mejor modelo:')
for example, pred_label in zip(manual_texts, manual_nb_pred):
    print('-' * 100)
    print(f"Tema esperado:  {example['tema_esperado']}")
    print(f"Clase predicha: {target_names[pred_label]}")
    print(f"Texto:          {preview_text(example['texto'], 260)}")


Clasificación Naïve Bayes de los textos manuales con el mejor modelo:
----------------------------------------------------------------------------------------------------
Tema esperado:  sci.space
Clase predicha: sci.space
Texto:          NASA announced a new mission to study distant planets and collect data from orbit. The spacecraft will use advanced instruments to analyze atmospheric conditions.
----------------------------------------------------------------------------------------------------
Tema esperado:  rec.autos
Clase predicha: rec.autos
Texto:          I am looking for advice about buying a used car. The engine runs well, but the brakes and transmission may need maintenance soon.
----------------------------------------------------------------------------------------------------
Tema esperado:  religion
Clase predicha: soc.religion.christian
Texto:          The discussion focused on belief, faith, and the existence of God. Several people debated moral arguments and religiou

**Interpretación del punto 3.** `ComplementNB` suele ser competitivo en clasificación de texto porque corrige parte del sesgo de `MultinomialNB` en clases desbalanceadas. Los cambios de vectorizador afectan mucho el resultado: remover stopwords, usar `min_df` y aplicar `sublinear_tf` cambian qué términos quedan como evidencia y cuánto pesa la frecuencia. La comparación debe centrarse en F1 macro porque todas las clases pesan igual, incluso las menos frecuentes.


### 4. Transponer la matriz documento-término y estudiar palabras similares

Al transponer `X_train`, cada fila pasa a representar una palabra y cada columna un documento. Dos palabras quedan cerca si aparecen en patrones de documentos parecidos. Esto no es exactamente semántica como en embeddings neuronales; es similaridad por coocurrencia documental dentro del corpus.


In [66]:
X_term_doc = X_train.T.tocsr()
manual_words = ['car', 'space', 'windows', 'bike', 'israel']

print(f'Matriz documento-término original: {X_train.shape}')
print(f'Matriz término-documento transpuesta: {X_term_doc.shape}')
print()

for word in manual_words:
    word_idx = tfidfvect.vocabulary_.get(word)

    if word_idx is None:
        print(f"La palabra '{word}' no está en el vocabulario.")
        continue

    sims = cosine_similarity(X_term_doc[word_idx], X_term_doc)[0]
    ordered = np.argsort(sims)[::-1]
    similar_word_indices = [i for i in ordered if i != word_idx][:5]

    print('=' * 80)
    print(f"Palabra elegida: '{word}'")
    for rank, similar_word_idx in enumerate(similar_word_indices, start=1):
        similar_word = idx2word[similar_word_idx]
        print(f'{rank}. {similar_word:<25} sim={sims[similar_word_idx]:.4f}')
    print()


Matriz documento-término original: (11314, 101631)
Matriz término-documento transpuesta: (101631, 11314)

Palabra elegida: 'car'
1. cars                      sim=0.1797
2. criterium                 sim=0.1770
3. civic                     sim=0.1748
4. owner                     sim=0.1689
5. dealer                    sim=0.1681

Palabra elegida: 'space'
1. nasa                      sim=0.3304
2. seds                      sim=0.2966
3. shuttle                   sim=0.2928
4. enfant                    sim=0.2803
5. seti                      sim=0.2465

Palabra elegida: 'windows'
1. dos                       sim=0.3037
2. ms                        sim=0.2320
3. microsoft                 sim=0.2219
4. nt                        sim=0.2140
5. for                       sim=0.1930

Palabra elegida: 'bike'
1. bikes                     sim=0.2767
2. ride                      sim=0.2716
3. cbr                       sim=0.2379
4. steet                     sim=0.2256
5. gsxr                      sim

**Interpretación del punto 4.** Las palabras similares deberían compartir contexto documental: por ejemplo, términos relacionados con autos deberían aparecer en documentos de `rec.autos`, y términos vinculados a espacio en `sci.space`. Si aparece una palabra rara o poco interpretable, probablemente se deba a ruido del corpus, nombres propios o tokens que coinciden en pocos documentos con mucho peso TF-IDF.


### Conclusión general

La representación TF-IDF permite resolver el desafío de dos maneras complementarias. La similaridad coseno sirve para recuperar documentos o palabras cercanas de forma interpretable. Los modelos Naïve Bayes agregan una capa supervisada que suele mejorar la clasificación porque combinan evidencia de muchos términos por clase, no solo el documento vecino más parecido.
